In [ ]:
from google.colab import files

uploaded = files.upload()  # A window will open to upload files

Saving mixtec-train.txt to mixtec-train.txt
Saving mixtec-val.txt to mixtec-val.txt
Saving spanish-train.txt to spanish-train.txt
Saving spanish-val.txt to spanish-val.txt


In [ ]:
# 📌 Install necessary libraries and upgrade if necessary
!pip install datasets transformers huggingface_hub --upgrade

# Importing required libraries
import pandas as pd
import json
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, TrainingArguments, Trainer, AutoModelForSeq2SeqLM
import wandb  # Weights & Biases for experiment tracking
import torch

# 📌 Read training files for Mixtec and Spanish sentences
with open("mixtec-train.txt", "r", encoding="utf-8") as f_mix_train, open("spanish-train.txt", "r", encoding="utf-8") as f_esp_train:
    mixteco_train_sentences = f_mix_train.readlines()
    espanol_train_sentences = f_esp_train.readlines()

# 📌 Assert that the training files have the same number of lines (sentences)
assert len(mixteco_train_sentences) == len(espanol_train_sentences), "The training files do not have the same number of lines."

# 📌 Read validation files for Mixtec and Spanish sentences
with open("mixtec-val.txt", "r", encoding="utf-8") as f_mix_val, open("spanish-val.txt", "r", encoding="utf-8") as f_esp_val:
    mixteco_val_sentences = f_mix_val.readlines()
    espanol_val_sentences = f_esp_val.readlines()

# 📌 Assert that the validation files have the same number of lines (sentences)
assert len(mixteco_val_sentences) == len(espanol_val_sentences), "The validation files do not have the same number of lines."

# 📌 Create DataFrames for training and validation data
train_df = pd.DataFrame({
    "spanish": [line.strip() for line in espanol_train_sentences],
    "mixtec": [line.strip() for line in mixteco_train_sentences]
})

val_df = pd.DataFrame({
    "spanish": [line.strip() for line in espanol_val_sentences],
    "mixtec": [line.strip() for line in mixteco_val_sentences]
})

# 📌 Save the DataFrames as JSON files for later use
train_dataset_path = "train_espanol_mixteco.json"
val_dataset_path = "val_espanol_mixteco.json"

train_df.to_json(train_dataset_path, orient="records", force_ascii=False, indent=4)
val_df.to_json(val_dataset_path, orient="records", force_ascii=False, indent=4)

# 📌 Print paths where datasets have been saved
print(f"Datasets saved at {train_dataset_path} and {val_dataset_path}")

# 📌 Load the saved datasets into the HuggingFace format
train_dataset = load_dataset("json", data_files=train_dataset_path)["train"]
val_dataset = load_dataset("json", data_files=val_dataset_path)["train"]

# 📌 Create a DatasetDict object to store the train and validation datasets
dataset = DatasetDict({
    "train": train_dataset,
    "test": val_dataset
})

# 📌 Load the pre-trained tokenizer and model (M2M100 model for multilingual translation)
model_name = "facebook/m2m100_418M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 🔄 Function to preprocess the dataset
def preprocess_function(examples):
    # Append </s> token to both input and target sentences
    inputs = [f"{text} </s>" for text in examples["spanish"]]
    targets = [f"{text} </s>" for text in examples["mixtec"]]

    # Tokenize the inputs and targets with padding and truncation
    model_inputs = tokenizer(inputs, padding="max_length", truncation=True, max_length=128)
    labels = tokenizer(targets, padding="max_length", truncation=True, max_length=128)

    # Set labels for the model
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 📌 Tokenize the entire dataset using the preprocess function
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# 📌 Set up the device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device)

# 📌 Set up training arguments for fine-tuning the model
training_args = TrainingArguments(
    output_dir="./results",  # Directory to save results
    evaluation_strategy="epoch",  # Evaluate after each epoch
    save_strategy="epoch",  # Save the model after each epoch
    per_device_train_batch_size=8,  # Set batch size for training (increased if enough memory)
    per_device_eval_batch_size=8,  # Set batch size for evaluation
    num_train_epochs=5,  # Number of training epochs (more if the data is limited)
    logging_dir="./logs",  # Directory for logs
    logging_steps=50,  # Log every 50 steps (to improve stability)
    learning_rate=3e-5,  # Learning rate (fine-tuned)
    weight_decay=0.01,  # Weight decay for regularization to avoid overfitting
    warmup_ratio=0.06,  # 6% of training for warmup phase
    fp16=True,  # Use 16-bit floating point precision if GPU is available
    report_to="wandb",  # Report metrics to Weights & Biases for experiment tracking
    push_to_hub=False  # Don't push the model to Hugging Face hub
)

# 📌 Initialize Weights & Biases for experiment tracking
wandb.login()
wandb.init(project="m2m100_finetuning_espanol_mixteco", name="m2m100_train_run")

# 📌 Create the Trainer instance to handle the training process
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
)

# 📌 Start the training process
trainer.train()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 120.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.0/468.0 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.27.1
    Uninstalling huggingface-hub-0.27.1:
      Successfully uninstalled huggingface-hub-0.27.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.47.1
    Uninstalling transformers-4.47.1:
      Successfully uninstalled transformers-4.47.1
Datasets guardados en train_espanol_mixteco.json y val_espanol_mi

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

...

wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hsantiago13 (hsantiago13-university-aut-noma-de-quer-taro). Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<ipython-input-2-9f30b83bb242>:99: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss
1,0.838000,0.224663
2,0.657000,0.208240
3,0.602400,0.197643
4,0.531400,0.196729
5,0.474100,0.196943


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:2810: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=7295, training_loss=0.8716156445111699, metrics={'train_runtime': 1117.5856, 'train_samples_per_second': 52.206, 'train_steps_per_second': 6.527, 'total_flos': 1.580496475127808e+16, 'train_loss': 0.8716156445111699, 'epoch': 5.0})

In [ ]:
# 📌 Save the fine-tuned model for Spanish to Mixtec translation
model.save_pretrained("./m2m100_modelo_espanol_mixteco")  # Save the model's weights and configuration
tokenizer.save_pretrained("./m2m100_modelo_espanol_mixteco")  # Save the tokenizer for the model

# 📌 Print a message to indicate that the training is complete and the model has been saved
print("🔹 Training complete and model saved at './m2m100_modelo_espanol_mixteco'.")

🔹 Entrenamiento completado y modelo guardado en './m2m100_modelo_espanol_mixteco'.


In [ ]:
# 📌 Install the necessary packages
!pip install sacrebleu pyter3

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import sacrebleu
import pyter
import json

# 📌 Load the fine-tuned model
model_name = "./m2m100_modelo_espanol_mixteco"  # Path where the trained model was saved
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 📌 Configure device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 📌 Load validation data
with open("spanish-val.txt", "r", encoding="utf-8") as f_esp_val, open("mixtec-val.txt", "r", encoding="utf-8") as f_mix_val:
    spanish_sentences = [line.strip() for line in f_esp_val.readlines()]
    reference_translations = [[line.strip() for line in f_mix_val.readlines()]]  # Nested list for sacrebleu

# Ensure the number of sentences in the Spanish and Mixtec validation files match
assert len(spanish_sentences) == len(reference_translations[0]), "Validation files have a different number of lines."

# 📌 Translation function
def translate(text):
    """Generates a translation using the fine-tuned model."""
    # Tokenize the input text and send to the appropriate device (GPU or CPU)
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    outputs = model.generate(**inputs, max_length=128)  # Generate translation
    return tokenizer.decode(outputs[0], skip_special_tokens=True)  # Decode the output

# 📌 Generate translations
predicted_translations = [translate(text) for text in spanish_sentences]

# 📌 Calculate BLEU score
bleu_score = sacrebleu.corpus_bleu(predicted_translations, reference_translations)
print(f"BLEU Score: {bleu_score.score}")

# 📌 Calculate TER (Translation Edit Rate)
def calculate_ter(hypotheses, references):
    # Calculate TER for each pair of hypothesis and reference
    ter_scores = [pyter.ter(hypothesis.split(), reference.split()) for hypothesis, reference in zip(hypotheses, references[0])]
    return sum(ter_scores) / len(ter_scores)

ter_score = calculate_ter(predicted_translations, reference_translations)
print(f"TER Score: {ter_score}")

BLEU Score: 2.6340062799652606
TER Score: 1.0595862902381687
